# Supply chain environment — combined diagnostic notebook

This notebook merges the former `hackathon/test_*.py` scripts with verbose **DEBUG** logging.

**Finding the repo:** the setup cell walks up from the current working directory until it finds `hackathon/server/app.py`, then adds the repo root and `hackathon/` to `sys.path`. Start Jupyter from anywhere inside the repo (e.g. repo root or `hackathon/test/`).

**Workflow**
1. Run **Setup** (imports + helpers).
2. Run **Test definitions** (loads all `test_*` functions; does not execute them).
3. Run **Run all in-process (1–6)** — or call `run_section("…", test_…)` yourself for one check.
4. Optionally run **HTTP client** (needs `uvicorn hackathon.server.app:app` on port 8000).

**Sections**
1. Network topology (7×3)
2. Legacy dimension expectations (likely fails vs current env)
3. News / disruption events
4. Sustainability (carbon)
5. Variable transport costs
6. Stochastic lead times
7. HTTP smoke test


## 1. Setup — path, imports, helpers


In [ ]:
from __future__ import annotations

import asyncio
import os
import sys
import traceback
from pathlib import Path
from typing import Any, Callable


def _find_repo_root(start: Path) -> Path:
    """Walk upward from cwd until hackathon/server/app.py exists."""
    cur = start.resolve()
    for _ in range(8):
        marker = cur / "hackathon" / "server" / "app.py"
        if marker.is_file():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError(
        "Could not find repo root (file hackathon/server/app.py). "
        f"Start Jupyter from the repo root or a subdirectory; cwd was {start!r}"
    )


REPO_ROOT = _find_repo_root(Path.cwd())
HACKATHON_DIR = REPO_ROOT / "hackathon"
for path in (str(REPO_ROOT), str(HACKATHON_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("DEBUG cwd:", Path.cwd())
print("DEBUG REPO_ROOT:", REPO_ROOT)
print("DEBUG HACKATHON_DIR:", HACKATHON_DIR)
print("DEBUG sys.path[0:3]:", sys.path[:3])

from hackathon.client import SupplyChainClient
from hackathon.models import AgentAction
from hackathon.server.hackathon_environment import SupplyChainEnv


def banner(title: str) -> None:
    line = "=" * 72
    print(f"\n{line}\n  {title}\n{line}")


def kv(name: str, value: Any) -> None:
    print(f"  {name}: {value!r}")


def run_section(name: str, fn: Callable[[], None]) -> None:
    banner(name)
    try:
        fn()
        print(f"\nDEBUG: section {name!r} finished without exception.")
    except Exception as exc:
        print(f"\n[FAIL] section {name!r} raised: {type(exc).__name__}: {exc}")
        traceback.print_exc()


def noop_action(q: float = 10.0) -> AgentAction:
    """Valid 7×3 action (21 lanes)."""
    return AgentAction(order_quantities=[q] * 21, shipping_methods=[0] * 21)


print("DEBUG: setup complete; imports OK.")


## 2. Test definitions (run after setup; no tests executed yet)


In [ ]:
def test_network_topology() -> None:
    env = SupplyChainEnv()
    print("DEBUG: SupplyChainEnv() constructed")
    kv("_num_nodes", env._num_nodes)
    kv("_num_products", env._num_products)
    kv("_num_methods", env._num_methods)
    kv("_horizon (default)", env._horizon)

    obs = env.reset(difficulty="mvp", seed=42)
    print("DEBUG: reset(difficulty='mvp', seed=42) done")
    kv("obs.day", obs.day)
    kv("len(inventory_levels)", len(obs.inventory_levels))
    kv("len(customer_backlog)", len(obs.customer_backlog))
    kv("len(state_vector)", len(obs.state_vector))
    kv("metadata keys", list(obs.metadata.keys()) if obs.metadata else [])

    exp_inv = 21
    if len(obs.inventory_levels) == exp_inv:
        print(f"[PASS] inventory_levels length == {exp_inv} (7×3)")
    else:
        print(f"[FAIL] inventory_levels length {len(obs.inventory_levels)} (expected {exp_inv})")

    exp_backlog = 12
    if len(obs.customer_backlog) == exp_backlog:
        print(f"[PASS] customer_backlog length == {exp_backlog} (4 retailers × 3 products)")
    else:
        print(f"[FAIL] customer_backlog length {len(obs.customer_backlog)} (expected {exp_backlog})")

    action = AgentAction(order_quantities=[10.0] * 21, shipping_methods=[0] * 21)
    print("DEBUG: stepping with 21 standard orders @ 10.0")
    obs = env.step(action)
    kv("obs.day after step", obs.day)
    kv("obs.reward", obs.reward)
    cd = obs.metadata.get("current_demands")
    kv("metadata['current_demands']", cd)
    if cd is not None:
        kv("len(current_demands)", len(cd))

    if cd is not None and len(cd) == 12:
        print("[PASS] current_demands length == 12")
    else:
        print(f"[FAIL] current_demands missing or wrong length: {cd!r}")

    exp_sv = 126
    if len(obs.state_vector) == exp_sv:
        print(f"[PASS] state_vector length == {exp_sv}")
    else:
        print(f"[FAIL] state_vector length {len(obs.state_vector)} (expected {exp_sv})")

    print("DEBUG: upstream routing _upstream_source(node) -> supplier index or None")
    tests = [(0, None), (1, 0), (2, 0), (3, 1), (4, 1), (5, 2), (6, 2)]
    for node, expected in tests:
        src = env._upstream_source(node)
        ok = src == expected
        tag = "[PASS]" if ok else "[FAIL]"
        print(f"  {tag} node {node}: upstream={src!r} expected={expected!r}")


def test_multi_product_legacy() -> None:
    env = SupplyChainEnv()
    obs = env.reset(difficulty="mvp")
    print("DEBUG: legacy test assumes 9 inventory slots, 3 backlog streams, 54-d state")
    kv("env._num_products", env._num_products)
    kv("len(obs.inventory_levels)", len(obs.inventory_levels))
    kv("len(obs.customer_backlog)", len(obs.customer_backlog))
    kv("len(obs.state_vector)", len(obs.state_vector))

    if len(obs.inventory_levels) == 9:
        print("[PASS] inventory_levels length 9 (legacy 3×3)")
    else:
        print(f"[FAIL] inventory_levels length {len(obs.inventory_levels)} (legacy expected 9)")

    if len(obs.customer_backlog) == 3:
        print("[PASS] customer_backlog length 3 (legacy)")
    else:
        print(f"[FAIL] customer_backlog length {len(obs.customer_backlog)} (legacy expected 3)")

    print("DEBUG: stepping with 21-lane AgentAction (required by current models)")
    action = AgentAction(order_quantities=[5.0] * 21, shipping_methods=[0] * 21)
    obs = env.step(action)
    kv("obs.day", obs.day)
    kv("obs.reward", obs.reward)
    cd = obs.metadata.get("current_demands")
    kv("current_demands", cd)

    if cd and len(cd) == 3:
        print("[PASS] current_demands length 3 (legacy)")
    else:
        print(f"[FAIL] current_demands legacy expected len 3, got {cd!r}")

    if len(obs.state_vector) == 54:
        print("[PASS] state_vector length 54 (legacy)")
    else:
        print(f"[FAIL] state_vector length {len(obs.state_vector)} (legacy expected 54)")


def test_news_events() -> None:
    env = SupplyChainEnv()
    env.reset(difficulty="mvp")
    print("DEBUG: manipulating _active_events for isolated checks")

    print("\n--- Social Media Trend (product 0 demand boost) ---")
    env._active_events = {"Social Media Trend": 2}
    demands = env._sample_customer_demand(day=1)
    kv("full demands len", len(demands))
    p0 = demands[0:4]
    kv("product 0 retailer demands", p0)
    if any(d > 40 for d in p0):
        print("[PASS] Social Media Trend: product 0 demands elevated (>40)")
    else:
        print("[FAIL] Social Media Trend: no demand > 40 in product 0 streams")

    print("\n--- Canal Blockage (lead time multiplier) ---")
    env._active_events = {"Canal Blockage": 2}
    action = AgentAction(order_quantities=[10.0] * 21, shipping_methods=[0] * 21)
    print("DEBUG: step with canal active; inspect warehouse A pipeline product 0")
    env.step(action)
    pipe = env._pipelines[1][0]
    kv("node 1 product 0 pipeline len", len(pipe))
    if pipe:
        latest_eta = pipe[-1].eta
        kv("latest shipment ETA", latest_eta)
        kv("base lead time node 1", env._lead_times[1])
        if latest_eta >= 5:
            print("[PASS] Canal Blockage: ETA >= 5 (increased vs base 3)")
        else:
            print("[FAIL] Canal Blockage: ETA not high enough")
    else:
        print("[FAIL] no pipeline shipment on node 1 p0 to inspect (empty pipeline)")

    print("\n--- Labor Strike (node 1 offline) ---")
    env._active_events = {"Labor Strike": 2}
    env._inventory[1] = [0.0] * 3
    env._order_backlogs[1] = [[10.0, 0.0] for _ in range(3)]
    print("DEBUG: dispatch with strike; node 1 inventory cleared, backlogs set")
    shipped = env._dispatch_replenishment_orders()
    node_1_slice = shipped[3:6]
    kv("shipped quantities indices 3:5 (node 1 products)", node_1_slice)
    node_1_shipped = sum(node_1_slice)
    if node_1_shipped == 0:
        print("[PASS] Labor Strike: no dispatch for node 1 products")
    else:
        print(f"[FAIL] Labor Strike: expected 0 shipped for node 1, got {node_1_shipped}")


def test_sustainability() -> None:
    env = SupplyChainEnv()
    obs = env.reset(difficulty="mvp", seed=0)
    kv("initial carbon_footprint (obs)", obs.carbon_footprint)

    print("DEBUG: step 1 — all standard shipping")
    action_std = AgentAction(order_quantities=[10.0] * 21, shipping_methods=[0] * 21)
    obs = env.step(action_std)
    carbon_std = obs.carbon_footprint
    kv("carbon after standard step", carbon_std)
    kv("reward_terms", obs.reward_terms)

    print("DEBUG: step 2 — all express shipping")
    action_exp = AgentAction(order_quantities=[10.0] * 21, shipping_methods=[1] * 21)
    obs = env.step(action_exp)
    carbon_exp = obs.carbon_footprint
    increase = carbon_exp - carbon_std
    kv("carbon after express step", carbon_exp)
    kv("delta carbon (express - prev)", increase)

    if increase > (carbon_std * 4):
        print("[PASS] express step added substantially more carbon than prior baseline")
    else:
        print(f"[FAIL] carbon increase {increase} not >> standard-era total {carbon_std}")

    pipe = env._pipelines[1][0]
    print("DEBUG: latest shipment to node 1 product 0")
    if not pipe:
        print("[FAIL] no shipments in node 1 p0 pipeline to verify express ETA")
        return
    latest = pipe[-1]
    kv("latest.method", latest.method)
    kv("latest.eta", latest.eta)
    if latest.method == 1 and latest.eta < 3:
        print("[PASS] express shipment has ETA < 3")
    elif latest.method == 0:
        print("[FAIL] latest shipment is standard (method 0), not express")
    else:
        print(f"[FAIL] express ETA {latest.eta} not < 3")


def test_variable_costs() -> None:
    env = SupplyChainEnv()
    env.reset(difficulty="mvp", seed=123)
    multipliers: list[float] = []
    transport_costs: list[float] = []

    print("DEBUG: 20 steps, identical orders; watch _fuel_price_multiplier and transport_cost")
    for i in range(20):
        action = AgentAction(order_quantities=[10.0] * 21, shipping_methods=[0] * 21)
        obs = env.step(action)
        multipliers.append(env._fuel_price_multiplier)
        tc = obs.reward_terms.get("transport_cost", float("nan"))
        transport_costs.append(tc)
        print(
            f"  day {i + 1:2d}  mult={env._fuel_price_multiplier:.4f}  "
            f"transport_cost={tc:.4f}  reward={obs.reward}"
        )

    um = set(round(m, 6) for m in multipliers)
    uc = set(round(t, 6) for t in transport_costs)
    print(f"DEBUG: unique multipliers (6dp): {len(um)} values")
    print(f"DEBUG: unique transport_costs (6dp): {len(uc)} values")
    kv("min/max multiplier", (min(multipliers), max(multipliers)))

    if len(um) > 1:
        print("[PASS] fuel multiplier varied over run")
    else:
        print("[FAIL] fuel multiplier never changed")

    if len(uc) > 1:
        print("[PASS] transport costs varied")
    else:
        print("[FAIL] transport costs constant")


def test_stochastic_lead_times() -> None:
    env = SupplyChainEnv()
    env.reset(difficulty="mvp", seed=7)
    kv("_lead_times", env._lead_times)

    collected: list[list[int]] = [[], [], []]
    print("DEBUG: 20 steps; record newest pipeline ETA for nodes 0..2 when pipeline non-empty")
    for i in range(20):
        obs = env.step(noop_action(10.0))
        for node_index in range(3):
            q = env._pipelines[node_index][0]
            if not q:
                continue
            newest = q[-1]
            collected[node_index].append(newest.eta)
            print(f"  step {i + 1:2d} node {node_index} newest ETA={newest.eta} method={newest.method}")

    for node_index in range(3):
        lts = collected[node_index]
        uniq = set(lts)
        kv(f"node {node_index} ETA samples count", len(lts))
        kv(f"node {node_index} unique ETAs", uniq)
        if len(uniq) > 1:
            print(f"[PASS] node {node_index}: stochastic ETAs {uniq}")
        elif len(lts) == 0:
            print(f"[FAIL] node {node_index}: no ETAs collected (pipelines often empty)")
        else:
            print(f"[FAIL] node {node_index}: only one ETA value observed: {uniq}")


async def test_http_client(base_url: str = "http://localhost:8000") -> None:
    print(f"DEBUG: connecting SupplyChainClient base_url={base_url!r}")
    env = SupplyChainClient(base_url=base_url)

    print("DEBUG: POST /reset")
    obs_raw = env.reset(difficulty="mvp", seed=0, horizon=365)
    if asyncio.iscoroutine(obs_raw):
        obs_raw = await obs_raw
    obs = obs_raw.observation if hasattr(obs_raw, "observation") else obs_raw
    print("DEBUG: reset response type", type(obs_raw).__name__)
    kv("obs.day", getattr(obs, "day", None))
    kv("len(inventory_levels)", len(getattr(obs, "inventory_levels", [])))

    print("DEBUG: POST /step express orders")
    action = AgentAction(order_quantities=[10.0] * 21, shipping_methods=[1] * 21)
    result_raw = env.step(action)
    if asyncio.iscoroutine(result_raw):
        result_raw = await result_raw

    result = result_raw.observation if hasattr(result_raw, "observation") else result_raw
    reward = getattr(result_raw, "reward", getattr(result, "reward", None))
    print("DEBUG: step result type", type(result_raw).__name__)
    kv("result.day", getattr(result, "day", None))
    kv("reward", reward)
    kv("carbon_footprint", getattr(result, "carbon_footprint", None))
    kv("active_events", getattr(result, "active_events", None))
    kv("fill_rate", getattr(result, "fill_rate", None))
    print("[PASS] HTTP reset + step completed without exception")


def run_all_in_process() -> None:
    tests = [
        ("1. Network topology", test_network_topology),
        ("2. Legacy multi-product dimensions", test_multi_product_legacy),
        ("3. News / disruption events", test_news_events),
        ("4. Sustainability", test_sustainability),
        ("5. Variable transport costs", test_variable_costs),
        ("6. Stochastic lead times", test_stochastic_lead_times),
    ]
    for name, fn in tests:
        run_section(name, fn)
    banner("Done: sections 1–6")
    print("Run the HTTP cell separately if the server is up.")


## 3. Run all in-process tests (sections 1–6)


In [ ]:
run_all_in_process()


## 4. HTTP client (optional; server on localhost:8000)


In [ ]:
await test_http_client()


## 5. Run a single section

After setup + definitions, execute in a **new code cell**:

```python
run_section("1. Network topology", test_network_topology)
# run_section("3. News / disruption events", test_news_events)
```
